# MOFA+ on per-drug PosteriorMean logFC matrices

Notebook version of `RunMOFA_PosteriorMatrices.py` — same pipeline, broken into
step-by-step cells so you can inspect intermediate state, swap parameters, and
add diagnostics inline.

**Setup.** Drugs are *views*, perturbations are *samples*, genes are *features*.
MOFA+ fits a single shared factor matrix `Z` ∈ ℝ^{n_perts × n_factors} and a
per-view loading matrix `W_v` ∈ ℝ^{n_genes × n_factors}, with ARD on both factors
and weights so unused (factor, view) pairs get zero loadings.

## Outputs (in `MOFA/`)
```
mofa_model.hdf5            full MOFA model (loadable via mofapy2)
factors.parquet            (n_perts, n_factors) factor matrix Z
loadings_<drug>.parquet    (n_genes, n_factors) per-view loadings W_v
factor_view_alpha.parquet  (n_factors, n_views) ARD α — small α = factor active in view
variance_explained.parquet per-(view, factor) R^2
axes.json                  drugs / perturbations / genes ordering
```


In [5]:
1.06 * 0.025

0.026500000000000003

## 1. Imports


In [1]:
import json, time, warnings
from pathlib import Path
from typing import Dict, List, Tuple

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


## 2. Configuration


In [2]:
PROJECT_DIR  = Path('/home/beraslan/Projects/ChemoGeneticScreens')
PM_DIR       = PROJECT_DIR / 'PosteriorMeanMatrices'
OUT_DIR      = PROJECT_DIR / 'MOFA'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model knobs
N_FACTORS         = 30      # initial number of factors; ARD prunes unused ones
GENE_MEDIAN_THR   = 0.0     # drop genes whose max-across-drugs |median logFC| is below this
                            # data is Bayesian-shrunk (|logFC| typically <0.01) — keep at 0
                            # to disable, or use a tiny value like 0.002 to drop dead genes
MAX_ITER          = 1000
SEED              = 0
SCALE_VIEWS       = False   # if True, MOFA scales variance-per-view to 1 before fitting

print(f'inputs:  {PM_DIR}')
print(f'outputs: {OUT_DIR}')


inputs:  /home/beraslan/Projects/ChemoGeneticScreens/PosteriorMeanMatrices
outputs: /home/beraslan/Projects/ChemoGeneticScreens/MOFA


## 3. Load per-drug `PosteriorMean_matrix_<drug>.csv`

Each file: rows = perturbations, columns = response genes (~18,154), values =
Bayesian-shrunk posterior-mean logFCs.


In [3]:
def load_per_drug(pm_dir: Path):
    files = sorted(pm_dir.glob('PosteriorMean_matrix_*.csv'))
    if not files:
        raise RuntimeError(f'No PosteriorMean_matrix_*.csv under {pm_dir}')
    drugs, mats = [], {}
    for fp in files:
        drug = fp.stem.replace('PosteriorMean_matrix_', '')
        drugs.append(drug)
        df = pd.read_csv(fp, index_col=0)
        df.index.name = 'perturbation'
        mats[drug] = df
        print(f'  {drug:30s}  {df.shape[0]:>5d} perts × {df.shape[1]:>6d} genes')
    return mats, drugs

mats, drugs = load_per_drug(PM_DIR)
print(f'\n{len(drugs)} drug contexts loaded')


  AR-A014418                       2294 perts ×  18154 genes
  AZD4573                          2248 perts ×  18154 genes
  Bisindolylmaleimide-I            2259 perts ×  18154 genes
  CHIR-98014                       2226 perts ×  18154 genes
  DG-172                           2307 perts ×  18154 genes
  DMSO_round2                      2212 perts ×  18154 genes
  DMSO_round2_batch2               2225 perts ×  18154 genes
  JTE-607                          2340 perts ×  18154 genes
  LDN-193189                       2293 perts ×  18154 genes
  LY2090314                        2201 perts ×  18154 genes
  Lexibulin                        2296 perts ×  18154 genes
  NSC95397                         2287 perts ×  18154 genes
  PP121                            2249 perts ×  18154 genes
  Romidepsin                       2288 perts ×  18154 genes
  Stattic                          2292 perts ×  18154 genes
  VX-11e                           2271 perts ×  18154 genes

16 drug contexts loaded

## 4. Align: intersect perturbations and genes across drugs

Optionally drop genes whose `|median logFC|` (taken per drug, then max across
drugs) is below `GENE_MEDIAN_THR`. The data is heavily shrunk (most |logFC| <
0.01), so the threshold should be tiny if used at all.


In [4]:
def align(mats, drugs, gene_median_thr):
    common_perts = sorted(set.intersection(*(set(df.index)   for df in mats.values())))
    common_genes = sorted(set.intersection(*(set(df.columns) for df in mats.values())))
    print(f'Intersection: {len(common_perts)} perts × {len(common_genes)} genes')

    if gene_median_thr > 0:
        per_drug_max_abs_med = np.zeros(len(common_genes))
        for d in drugs:
            v = mats[d].loc[common_perts, common_genes].abs().median(axis=0).values
            per_drug_max_abs_med = np.maximum(per_drug_max_abs_med, v)
        keep = per_drug_max_abs_med >= gene_median_thr
        common_genes = [g for g, k in zip(common_genes, keep) if k]
        print(f'After |median logFC| ≥ {gene_median_thr}: {len(common_genes)} genes kept')

    aligned = {d: mats[d].loc[common_perts, common_genes] for d in drugs}
    return aligned, common_perts, common_genes

aligned, perts, genes = align(mats, drugs, GENE_MEDIAN_THR)
print(f'\nFinal aligned tensor: {len(perts)} perts × {len(genes)} genes × {len(drugs)} views')


Intersection: 2124 perts × 18154 genes

Final aligned tensor: 2124 perts × 18154 genes × 16 views


### Quick sanity-check on the data scale

Posterior-mean logFCs are Bayesian-shrunk so magnitudes are small. Confirm the
distribution before fitting Gaussian-likelihood MOFA.


In [ ]:
sample_drug = drugs[0]
v = aligned[sample_drug].values
print(f'{sample_drug}: shape={v.shape}, mean={v.mean():.4f}, std={v.std():.4f}')
print(f'  |logFC| quantiles 50/90/99/max:  '
      f'{np.quantile(np.abs(v), [.5,.9,.99]).round(4)}  max={np.abs(v).max():.3f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(v.flatten(), bins=200, log=True)
ax.set_xlabel('logFC'); ax.set_ylabel('count (log)'); ax.set_title(f'{sample_drug} value distribution')
plt.show()


## 5. Build the MOFA+ data structure

`mofapy2` expects `data[v][g]` = numpy `(n_samples × n_features)` matrix per
(view, group). We have one sample group, so g=0; one view per drug.


In [ ]:
def to_mofa_inputs(aligned, drugs):
    data = [[aligned[d].values.astype(np.float32)] for d in drugs]
    sample_names  = list(aligned[drugs[0]].index)
    feature_names = list(aligned[drugs[0]].columns)
    return data, sample_names, feature_names

data, sample_names, feature_names = to_mofa_inputs(aligned, drugs)
print(f'data is a list of {len(data)} views, each list of 1 group')
print(f'view 0 ({drugs[0]}): {data[0][0].shape}')


## 6. Configure MOFA+

Key choices:

- `likelihoods='gaussian'` — explicitly set; MOFA's auto-inference can pick
  `bernoulli` when most values are near zero (which our shrunken logFCs are).
- `spikeslab_weights=True` — sparsity prior on gene loadings.
- `ard_factors=True` — per-(view, factor) variance prior. **This is what gives
  shared/private decomposition for free.** Small α = factor active in view;
  large α = loading shrunk toward zero.
- `ard_weights=True` — column-level sparsity on top of spike-and-slab.
- `convergence_mode='medium'` — tightness of the ELBO stopping criterion.


In [ ]:
from mofapy2.run.entry_point import entry_point

ent = entry_point()
ent.set_data_options(scale_views=SCALE_VIEWS, scale_groups=False)
ent.set_data_matrix(
    data,
    views_names=drugs,
    groups_names=['all'],
    samples_names=[sample_names],
    features_names=[feature_names for _ in drugs],
    likelihoods=['gaussian'] * len(drugs),
)
ent.set_model_options(
    factors=N_FACTORS,
    likelihoods=['gaussian'] * len(drugs),
    spikeslab_weights=True,
    ard_factors=True,
    ard_weights=True,
)
ent.set_train_options(
    iter=MAX_ITER,
    convergence_mode='medium',
    startELBO=1,
    freqELBO=10,
    gpu_mode=False,
    seed=SEED,
    verbose=True,
)
print('MOFA+ entry point configured')


## 7. Build & run

This is the long step. Expect 5–20 minutes depending on data size and
convergence. Watch for ELBO updates printed by MOFA — should monotonically
increase and stabilize.


In [ ]:
t0 = time.time()
ent.build()
ent.run()
print(f'\ntrained in {(time.time()-t0)/60:.1f} min')


## 8. Save the canonical HDF5 model

`save_data=False` keeps the file small — we already have the input data in CSVs.


In [ ]:
model_path = OUT_DIR / 'mofa_model.hdf5'
ent.save(str(model_path), save_data=False)
print(f'Saved model → {model_path}  ({model_path.stat().st_size/1e6:.1f} MB)')


## 9. Pull tabular outputs from the HDF5

- `expectations/Z`            → factor matrix `(n_samples, n_factors)`
- `expectations/W`            → per-view loading matrix `(n_features, n_factors)`
- `expectations/AlphaW`       → per-(view, factor) ARD α (large α = factor inactive in view)
- `variance_explained/r2_per_factor` → per-(view, factor) R² (variance explained)


In [ ]:
with h5py.File(model_path, 'r') as h:
    z_keys = list(h['expectations/Z'].keys())
    Z = np.asarray(h[f'expectations/Z/{z_keys[0]}'][()]).T              # (n_samples, n_factors)
    view_keys = list(h['expectations/W'].keys())
    W = {v: np.asarray(h[f'expectations/W/{v}'][()]).T for v in view_keys}
    alpha_keys = list(h['expectations/AlphaW'].keys())
    alpha_per_view = {v: np.asarray(h[f'expectations/AlphaW/{v}'][()]).flatten()
                       for v in alpha_keys}

factor_cols = [f'factor_{i}' for i in range(Z.shape[1])]
print(f'Z (factor matrix): {Z.shape}')
print(f'W per view:        {[w.shape for w in W.values()][:2]} ... ({len(W)} views)')
print(f'AlphaW per view:   {next(iter(alpha_per_view.values())).shape}')


## 10. Save factor matrix and per-view loadings as parquet


In [ ]:
factor_df = pd.DataFrame(Z, index=sample_names, columns=factor_cols)
factor_df.index.name = 'perturbation'
factor_df.to_parquet(OUT_DIR / 'factors.parquet')
print(f'factors.parquet  ({factor_df.shape[0]} perts × {factor_df.shape[1]} factors)')

for v, mat in W.items():
    Wdf = pd.DataFrame(mat, index=feature_names, columns=factor_cols)
    Wdf.index.name = 'gene'
    Wdf.to_parquet(OUT_DIR / f'loadings_{v}.parquet')
print(f'loadings_<drug>.parquet × {len(W)} files')


## 11. Per-(factor, view) ARD α

This is the key output for shared-vs-private interpretation. Build a
`(n_factors, n_views)` matrix where each entry is α — **small = factor active
in that view, large = loadings shrunk toward zero.**


In [ ]:
alpha_df = pd.DataFrame(
    np.column_stack([alpha_per_view[v] for v in drugs]),
    index=factor_cols, columns=drugs,
)
alpha_df.index.name = 'factor'
alpha_df.to_parquet(OUT_DIR / 'factor_view_alpha.parquet')
print(alpha_df.round(2).to_string())


## 12. Variance explained per (view, factor)


In [ ]:
with h5py.File(model_path, 'r') as h:
    if 'variance_explained' in h:
        r2 = {}
        for v in drugs:
            if v in h['variance_explained/r2_per_factor']:
                g_keys = list(h[f'variance_explained/r2_per_factor/{v}'].keys())
                r2[v] = np.asarray(h[f'variance_explained/r2_per_factor/{v}/{g_keys[0]}'][()]).flatten()
        if r2:
            r2_df = pd.DataFrame(
                np.column_stack([r2[v] for v in drugs]),
                index=factor_cols, columns=drugs,
            )
            r2_df.index.name = 'factor'
            r2_df.to_parquet(OUT_DIR / 'variance_explained.parquet')
            print(r2_df.round(3).to_string())
        else:
            r2_df = None
            print('variance_explained absent from model file')
    else:
        r2_df = None
        print('variance_explained absent from model file')


## 13. Visualize factor activity across views

Two heatmaps:

- `log10(α)` per (factor, view) — low (blue) = factor active in that view
- variance explained R² per (factor, view) — what fraction of view variance the factor captures


In [ ]:
fig, axes = plt.subplots(1, 2 if r2_df is not None else 1,
                          figsize=(16 if r2_df is not None else 8, max(6, 0.25*N_FACTORS+1)))
if r2_df is None:
    axes = [axes]
sns.heatmap(np.log10(alpha_df), ax=axes[0], cmap='viridis_r',
            cbar_kws={'label': 'log10(α)  (low = active)'},
            xticklabels=True, yticklabels=True)
axes[0].set_title('Per-(factor, view) ARD α  (log10)')
axes[0].set_ylabel('factor'); axes[0].set_xlabel('drug')

if r2_df is not None:
    sns.heatmap(r2_df, ax=axes[1], cmap='magma',
                cbar_kws={'label': 'R²'},
                xticklabels=True, yticklabels=True)
    axes[1].set_title('Variance explained R² per (factor, view)')
    axes[1].set_ylabel(''); axes[1].set_xlabel('drug')

plt.tight_layout(); plt.savefig(OUT_DIR / 'factor_activity_heatmaps.png', dpi=120, bbox_inches='tight'); plt.show()


## 14. Save axes metadata

Drug order, perturbation list, gene list — useful for any downstream code that
needs to align matrices.


In [ ]:
with open(OUT_DIR / 'axes.json', 'w') as f:
    json.dump({
        'drugs':         drugs,
        'perturbations': sample_names,
        'genes':         feature_names,
        'n_factors':     int(Z.shape[1]),
        'gene_median_thr': GENE_MEDIAN_THR,
        'seed':          SEED,
    }, f, indent=2)
print(f'axes.json  ({len(sample_names)} perts, {len(feature_names)} genes, {len(drugs)} views)')


## 15. Classify factors as shared / private / partial

A factor is **active in a view** if its α is below `2 × median(α)` (loadings
un-shrunken). Then:

- **Shared** — active in ≥ K−1 of K views
- **Private** — active in exactly 1 view
- **Partial** — active in 2 to K−2 views
- **Dead** — active in 0 views (ARD pruned)


In [ ]:
alpha_arr = alpha_df.values                  # (n_factors, n_views)
thr = 2.0 * np.median(alpha_arr)
active = alpha_arr < thr                     # (n_factors, n_views)
n_active = active.sum(axis=1)

def label_factor(n):
    if n == 0:           return 'dead'
    if n == 1:           return 'private'
    if n >= len(drugs)-1: return 'shared'
    return f'partial ({n}/{len(drugs)})'

summary = pd.DataFrame({
    'factor':         factor_cols,
    'n_active_views': n_active,
    'label':          [label_factor(n) for n in n_active],
    'min_alpha':      alpha_arr.min(1),
    'max_alpha':      alpha_arr.max(1),
    'active_views':   [','.join(d for d, a in zip(drugs, row) if a) for row in active],
})
summary.to_csv(OUT_DIR / 'factor_summary.csv', index=False)
print(summary.to_string(index=False))
print()
print(f'Counts:  shared={(n_active >= len(drugs)-1).sum()},  '
      f'private={(n_active == 1).sum()},  '
      f'partial={((n_active>=2)&(n_active<=len(drugs)-2)).sum()},  '
      f'dead={(n_active==0).sum()}')


## 16. Top-loading genes per factor

For each factor, look at the loadings in the view where it's most active (smallest α).
Top genes are the directly readable gene program for that factor in that view.


In [ ]:
def top_loadings(W_v_arr, gene_names, k, n_top=15):
    w = W_v_arr[:, k]
    up = np.argsort(w)[-n_top:][::-1]
    dn = np.argsort(w)[:n_top]
    return pd.DataFrame({
        'top_up':   [gene_names[i] for i in up],
        'up_w':     w[up].round(3),
        'top_dn':   [gene_names[i] for i in dn],
        'dn_w':     w[dn].round(3),
    })

# Pick the first 5 non-dead factors to inspect
non_dead = [k for k in range(N_FACTORS) if n_active[k] > 0]
for k in non_dead[:5]:
    v_star = drugs[int(np.argmin(alpha_arr[k]))]
    W_v_star = W[v_star]
    print(f'\n=== factor_{k}  ({summary.loc[k, "label"]})  '
          f'most active in {v_star} (α={alpha_arr[k].min():.2f}) ===')
    print(top_loadings(W_v_star, feature_names, k, n_top=10).to_string(index=False))


## 17. (Optional) Visualize factor matrix Z by drug

Z is `(n_perts, n_factors)`. The same pert appears once in Z (the joint
representation), but you can colour by KO target or any pert metadata.


In [ ]:
# Quick scatter: factor 0 vs factor 1, points = perturbations
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Z[:, 0], Z[:, 1], s=8, alpha=0.5)
ax.set_xlabel('factor 0'); ax.set_ylabel('factor 1')
ax.set_title(f'MOFA factor matrix Z ({Z.shape[0]} perts)')
ax.grid(alpha=0.3)
plt.show()


## 18. (Next steps)

From here you can:

1. **ORA on top genes per factor** — feed top-loading gene lists into the same
   hypergeometric ORA used in notebooks 15–17 to label what each factor *is*.
2. **Compare with the deep VAE variants** — the activity matrix from
   `14_3_VAEModel_singleblock` (`factor_view_activity.parquet`) is the deep
   analogue of `factor_view_alpha.parquet`. Both should agree on which factors
   are shared vs private; disagreements highlight non-linear structure that the
   linear MOFA misses.
3. **Pert metadata overlay** — colour Z by KO target / pathway membership to see
   whether known biology clusters in factor space.
4. **Drug-context interpretation** — for partial / private factors, the active
   views indicate which drug contexts share that program — these are candidate
   drug-class programs.
